In [1]:
import torch
import sys
import os
from torch.utils.data import DataLoader

In [2]:
if os.path.exists("/nas/longleaf/home/emprzy"):
    project_root = "/nas/longleaf/home/emprzy/influpaint"
    # This matches the directory you just created
    data_path = os.path.join(project_root, "training_datasets/TS_30S70M_2025-07-17.nc")
else:
    project_root = "/Users/emprzy/Documents/work/influpaint"
    data_path = "/Users/emprzy/Documents/work/miscellaneous/influpaint_data/TS_30S70M_2025-07-17.nc"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Point to the specific CoPaint folder
copaint_path = os.path.join(project_root, "influpaint/batch/CoPaint4influpaint")
if copaint_path not in sys.path:
    sys.path.insert(0, copaint_path)

print(f"Environment: {'Longleaf' if 'nas' in project_root else 'Mac'}")
print(f"Project root: {project_root}")
print(f"Data path: {data_path}")

Environment: Mac
Project root: /Users/emprzy/Documents/work/influpaint
Data path: /Users/emprzy/Documents/work/miscellaneous/influpaint_data/TS_30S70M_2025-07-17.nc


In [3]:
from influpaint.batch.scenarios import get_training_scenario, create_scenario_objects, print_available_scenarios
from influpaint.batch.config import transform_library
from influpaint.datasets import loaders as training_datasets

In [5]:
# season_setup = SeasonAxis.for_flusight(remove_us=True, remove_territories=True) 
image_size = 64
channels = 6
batch_size=512
epochs=3000
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
scn_id = 868  # i868::m_U500cRx1224::ds_30S70M::tr_Sqrt::ri_No
experiment_name = "emily_first_train"  # MLflow experiment name
scenario_spec = get_training_scenario(scn_id)
ddpm, dataset, transform, enrich, scaling_per_channel, data_mean, data_sd = create_scenario_objects(
            scenario_spec, image_size, channels, batch_size, epochs, device) # PATCH: removed season_setup param (don't need it b/c it is only used for `datasets`, which i will set explicitly)
# dataset = training_datasets.FluDataset.from_xarray("/Users/emprzy/Documents/work/miscellaneous/influpaint_data/TS_30S70M_2025-07-17.nc",channels=channels,)
# don't need ^this^ line because i modified create_scenario_objects()

created dataset with max [188028.  109653.8 109653.8 109653.8 109653.8 109653.8], full dataset has shape (10000, 6, 64, 64)
test passed: back and forth transformation are ok ✅


In [ ]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
losses = ddpm.train(dataloader, mlflow_logging=True)